# Chapter 15: Time Series Data Handling

**Companion notebook** for *Beginner's Guide to Pandas* by Ravi Shankar

Run each cell in order. Exercises are at the end.

In [1]:
import pandas as pd
import numpy as np

# Time Series Data Handling

Time series data is everywhere—stock prices, sensor readings, website traffic, weather measurements. Pandas provides powerful tools to work with temporal data efficiently. This chapter covers the essential techniques for handling datetime types, parsing and indexing dates, resampling time series data, analyzing patterns with rolling windows, and manipulating dates with offsets and time zones.

---

## Understanding Datetime Types in Pandas

### Creating Datetime Objects

Pandas represents dates and times using the `datetime64[ns]` dtype, which stores timestamps with nanosecond precision. You can create datetime objects in several ways:

In [2]:
import pandas as pd
import numpy as np

# Create a single timestamp
ts = pd.Timestamp('2024-01-15 10:30:00')
print(ts)
print(type(ts))

# Create a series of timestamps
dates = pd.date_range('2024-01-01', periods=5, freq='D')
print(dates)

# Convert strings to datetime
date_series = pd.to_datetime(['2024-01-01', '2024-01-02', '2024-01-03'])
print(date_series)

2024-01-15 10:30:00
<class 'pandas._libs.tslibs.timestamps.Timestamp'>
DatetimeIndex(['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04',
               '2024-01-05'],
              dtype='datetime64[ns]', freq='D')
DatetimeIndex(['2024-01-01', '2024-01-02', '2024-01-03'], dtype='datetime64[ns]', freq=None)


A **`pd.Timestamp`** is a single point in time (an enhanced version of Python's `datetime`), while a **`pd.DatetimeIndex`** is an index of many timestamps optimized for time series operations. Once you have a DatetimeIndex, you can extract components directly:

In [3]:
ts = pd.Timestamp('2023-01-15')
print(ts.year, ts.month, ts.day)   # 2023 1 15
print(ts.day_name())                # Sunday

2023 1 15
Sunday


### Why Parse Dates Early?

When dates are stored as strings, pandas treats them as text, which causes sorting and filtering errors. Parsing dates on import is faster and unlocks time-aware operations:

In [4]:
import pandas as pd
from io import StringIO

data_unparsed = """date,price
2024-01-15,100
2024-01-10,95
2024-01-20,105"""

# Without parsing: sorts alphabetically (wrong)
df_unparsed = pd.read_csv(StringIO(data_unparsed))
print("Sorted unparsed (WRONG - alphabetical order):")
print(df_unparsed.sort_values('date'))

# With parsing: sorts chronologically (correct)
df_parsed = pd.read_csv(StringIO(data_unparsed), parse_dates=['date'])
print("\nSorted parsed (CORRECT - chronological order):")
print(df_parsed.sort_values('date'))
print(f"Data type: {df_parsed['date'].dtype}")

Sorted unparsed (WRONG - alphabetical order):
         date  price
1  2024-01-10     95
0  2024-01-15    100
2  2024-01-20    105

Sorted parsed (CORRECT - chronological order):
        date  price
1 2024-01-10     95
0 2024-01-15    100
2 2024-01-20    105
Data type: datetime64[ns]


**Key benefits of early parsing:**
- Correct sorting and filtering
- Faster performance (parsed once, not repeatedly)
- Access to date-specific operations via the `.dt` accessor
- Enables resampling and time-series analysis

### Setting Dates as Index

For resampling and time-based slicing, set the datetime column as the index:

In [5]:
import pandas as pd
from io import StringIO

csv_data = """date,value
2023-01-01,100
2023-01-02,102
2023-01-03,101"""

df = pd.read_csv(StringIO(csv_data), parse_dates=['date'])
df.set_index('date', inplace=True)
print(df)
print(type(df.index))  # <class 'pandas.core.indexes.datetimes.DatetimeIndex'>

# Date-based slicing works naturally
print(df['2023-01-01':'2023-01-02'])

            value
date             
2023-01-01    100
2023-01-02    102
2023-01-03    101
<class 'pandas.core.indexes.datetimes.DatetimeIndex'>
            value
date             
2023-01-01    100
2023-01-02    102


**Key insight:** A DatetimeIndex enables pandas to optimize operations. Regular indices don't understand temporal relationships—DatetimeIndex does.

### Handling Different Date Formats

In [6]:
import pandas as pd

# US format (MM/DD/YYYY)
us_dates = pd.to_datetime(['01/15/2024', '01/10/2024'], format='%m/%d/%Y')
print("US format:", us_dates)

# European format (DD/MM/YYYY)
eu_dates = pd.to_datetime(['15/01/2024', '10/01/2024'], format='%d/%m/%Y')
print("EU format:", eu_dates)

# Custom format with time
custom = pd.to_datetime(['15-Jan-2024 14:30:00', '10-Jan-2024 09:15:00'],
                        format='%d-%b-%Y %H:%M:%S')
print("Custom format:", custom)

# Handle invalid dates gracefully
dates_with_errors = ['2024-01-15', '2024-13-45', 'invalid', '2024-02-29']
result = pd.to_datetime(dates_with_errors, errors='coerce')
print("With errors='coerce':", result)

US format: DatetimeIndex(['2024-01-15', '2024-01-10'], dtype='datetime64[ns]', freq=None)
EU format: DatetimeIndex(['2024-01-15', '2024-01-10'], dtype='datetime64[ns]', freq=None)
Custom format: DatetimeIndex(['2024-01-15 14:30:00', '2024-01-10 09:15:00'], dtype='datetime64[ns]', freq=None)
With errors='coerce': DatetimeIndex(['2024-01-15', 'NaT', 'NaT', '2024-02-29'], dtype='datetime64[ns]', freq=None)


### Assembling Datetimes from Components

When your data has separate year, month, day, and time columns, you can assemble them into a single datetime column:

In [7]:
import pandas as pd

df = pd.DataFrame({
    'year': [2024, 2024, 2024],
    'month': [1, 2, 3],
    'day': [15, 20, 10],
    'hour': [9, 14, 16],
    'minute': [30, 45, 0]
})

df['datetime'] = pd.to_datetime(df[['year', 'month', 'day', 'hour', 'minute']])
print(df)

   year  month  day  hour  minute            datetime
0  2024      1   15     9      30 2024-01-15 09:30:00
1  2024      2   20    14      45 2024-02-20 14:45:00
2  2024      3   10    16       0 2024-03-10 16:00:00


### The `.dt` Accessor

The `.dt` accessor provides access to datetime properties on a Series column:

In [8]:
import pandas as pd
from io import StringIO

data = """date,price
2024-01-15,100
2024-03-10,95
2024-06-20,105
2024-12-25,110"""

stocks = pd.read_csv(StringIO(data), parse_dates=['date'])

print("Year:", stocks['date'].dt.year.values)
print("Month:", stocks['date'].dt.month.values)
print("Day:", stocks['date'].dt.day.values)
print("Day name:", stocks['date'].dt.day_name().values)
print("Quarter:", stocks['date'].dt.quarter.values)

# Extract components into new columns
stocks['year'] = stocks['date'].dt.year
stocks['month_name'] = stocks['date'].dt.month_name()
stocks['day_name'] = stocks['date'].dt.day_name()
print(stocks)

Year: [2024 2024 2024 2024]
Month: [ 1  3  6 12]
Day: [15 10 20 25]
Day name: ['Monday' 'Sunday' 'Thursday' 'Wednesday']
Quarter: [1 1 2 4]
        date  price  year month_name   day_name
0 2024-01-15    100  2024    January     Monday
1 2024-03-10     95  2024      March     Sunday
2 2024-06-20    105  2024       June   Thursday
3 2024-12-25    110  2024   December  Wednesday


### Date-Based Slicing and Filtering

In [9]:
import pandas as pd

dates = pd.date_range('2023-01-01', periods=365, freq='D')
prices = pd.Series(range(100, 465), index=dates)

# Slice by date range (both inclusive)
subset = prices['2023-01-01':'2023-01-31']
print(len(subset))  # 31

# Partial string indexing: all of January
jan_2023 = prices['2023-01']
print(len(jan_2023))  # 31

# Filter using boolean indexing
df = pd.DataFrame({
    'date': pd.date_range('2023-01-01', periods=365, freq='D'),
    'value': range(365)
})

# All Mondays
mondays = df[df['date'].dt.day_name() == 'Monday']
print(len(mondays))  # ~52

# Date range filter
mask = (df['date'] >= '2023-01-01') & (df['date'] <= '2023-03-31')
q1 = df[mask]
print(len(q1))

31
31
52
90


### Date Arithmetic with Timedeltas

In [10]:
import pandas as pd
from io import StringIO

data = """date,price
2024-01-15,100
2024-03-10,95
2024-06-20,105"""

stocks = pd.read_csv(StringIO(data), parse_dates=['date'])

stocks['date_plus_30'] = stocks['date'] + pd.Timedelta(days=30)
stocks['date_minus_7'] = stocks['date'] - pd.Timedelta(days=7)
stocks['days_since_2024_start'] = (stocks['date'] - pd.Timestamp('2024-01-01')).dt.days

print(stocks[['date', 'date_plus_30', 'date_minus_7', 'days_since_2024_start']])

# Duration between two dates
start = pd.Timestamp('2024-01-01')
end = pd.Timestamp('2024-01-31')
duration = end - start
print(f"Duration: {duration.days} days")

        date date_plus_30 date_minus_7  days_since_2024_start
0 2024-01-15   2024-02-14   2024-01-08                     14
1 2024-03-10   2024-04-09   2024-03-03                     69
2 2024-06-20   2024-07-20   2024-06-13                    171
Duration: 30 days


---

## Frequency Strings: The Offset Alias Reference

When working with time series, you specify frequencies using **offset aliases**. Understanding these is crucial for resampling and date arithmetic.

| Alias | Meaning | Notes |
|-------|---------|-------|
| `'D'` | Calendar day | Every day |
| `'B'` | Business day | Monday–Friday only |
| `'W'` | Weekly | Ends on Sunday by default |
| `'ME'` | Month-end | Last day of month (pandas 2.0+) |
| `'MS'` | Month-start | First day of month |
| `'QE'` | Quarter-end | Last day of quarter (pandas 2.0+) |
| `'QS'` | Quarter-start | First day of quarter |
| `'YE'` | Year-end | December 31 (pandas 2.0+) |
| `'YS'` | Year-start | January 1 |
| `'h'` | Hourly | Every hour |
| `'min'` | Minute | Every minute |
| `'s'` | Second | Every second |

**Note:** Pandas 2.0+ uses `'ME'`, `'QE'`, `'YE'` (replacing `'M'`, `'Q'`, `'Y'`), `'h'` (replacing `'H'`), and `'min'` (replacing `'T'`).

You can combine numbers with aliases to create custom frequencies:

In [ ]:
import pandas as pd

# 2-day frequency
dates = pd.date_range('2023-01-01', periods=5, freq='2D')
print(dates)

# Every 15 minutes
dates = pd.date_range('2023-01-01', periods=5, freq='15min')
print(dates)

# Business days only
dates = pd.date_range('2023-01-01', periods=5, freq='B')
print(dates)

# Month-end vs. month-start
dates_end = pd.date_range('2023-01-01', periods=3, freq='ME')
print("Month-end:", dates_end)

dates_start = pd.date_range('2023-01-01', periods=3, freq='MS')
print("Month-start:", dates_start)

---

## Resampling Time Series Data

Resampling changes the frequency of your time series. You might aggregate daily data into monthly summaries (downsampling) or interpolate monthly data into daily observations (upsampling).

**Important:** Resampling requires a **DatetimeIndex**.

### Downsampling: Aggregating to Lower Frequency

Downsampling combines multiple observations into fewer ones using an aggregation function:

In [ ]:
import pandas as pd
import numpy as np

# Create hourly data
hourly_data = pd.Series(
    [10, 12, 15, 14, 18, 20, 19, 22, 25, 23, 21, 24],
    index=pd.date_range('2024-01-01', periods=12, freq='h')
)

print("Original hourly data:")
print(hourly_data)

# Downsample to 3-hourly using different aggregation functions
print("\nSum every 3 hours:")
print(hourly_data.resample('3h').sum())

print("\nMean every 3 hours:")
print(hourly_data.resample('3h').mean())

print("\nMax every 3 hours:")
print(hourly_data.resample('3h').max())

**Intuition:** Think of downsampling as "bucketing" your data. You group hourly values into 3-hour buckets and compute a statistic for each bucket.

### Common Aggregation Methods

| Method | When to Use | Example |
|--------|-------------|---------|
| `.mean()` | Average values | Daily prices → monthly average |
| `.sum()` | Total quantities | Daily volumes → weekly total |
| `.last()` | Final value in period | Daily closes → weekly close |
| `.first()` | Opening value | Daily opens → monthly open |
| `.min()` / `.max()` | Extremes | Daily highs → monthly high |
| `.std()` | Volatility | Daily returns → monthly volatility |
| `.ohlc()` | Open-High-Low-Close | Daily OHLC → weekly OHLC |

### Multiple Aggregations at Once

In [ ]:
import pandas as pd
import numpy as np

# Create intraday trading data
times = pd.date_range('2024-01-01 09:30', periods=100, freq='5min')
prices = 100 + np.random.randn(100).cumsum()
volume = np.random.randint(1000, 5000, 100)

df = pd.DataFrame({'price': prices, 'volume': volume}, index=times)

# Different aggregations for different columns
hourly = df.resample('h').agg({
    'price': 'last',    # Last price of the hour
    'volume': 'sum'     # Total volume
})
print(hourly)

# Multiple statistics for one column
price_stats = df['price'].resample('h').agg(['mean', 'min', 'max', 'std'])
print(price_stats)

### Upsampling: Interpolating to Higher Frequency

Upsampling increases the frequency, creating new timestamps that need to be filled:

In [ ]:
import pandas as pd

# Create daily data
daily_data = pd.Series(
    [100, 110, 105, 115],
    index=pd.date_range('2024-01-01', periods=4, freq='D')
)

print("Original daily data:")
print(daily_data)

# Upsample to 6-hourly frequency
print("\nUpsampled (no filling):")
print(daily_data.resample('6h').asfreq())

print("\nForward fill:")
print(daily_data.resample('6h').ffill())

print("\nBackward fill:")
print(daily_data.resample('6h').bfill())

print("\nLinear interpolation:")
print(daily_data.resample('6h').interpolate())

### Controlling Bin Edges and Labels

When resampling, you can control which side of the time interval is closed and how bins are labeled:

In [ ]:
import pandas as pd

minute_data = pd.Series(
    range(9),
    index=pd.date_range('2024-01-01', periods=9, freq='min')
)

# Default: left-closed, left-labeled
print("Default (left-closed, left-labeled):")
print(minute_data.resample('3min').sum())

# Right-closed, right-labeled
print("\nRight-closed, right-labeled:")
print(minute_data.resample('3min', closed='right', label='right').sum())

### Resampling DataFrames

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'temperature': [20.1, 20.5, 21.2, 21.8, 22.1, 22.5],
    'humidity': [65, 68, 70, 72, 71, 69],
    'pressure': [1013, 1013.5, 1014, 1014.2, 1014.1, 1013.8]
}, index=pd.date_range('2024-01-01', periods=6, freq='h'))

print("Resampled to 2-hourly (mean):")
print(df.resample('2h').mean())

print("\nCustom aggregation per column:")
print(df.resample('2h').agg({
    'temperature': 'mean',
    'humidity': 'max',
    'pressure': 'min'
}))

### Handling Missing Data During Resampling

In [17]:
import pandas as pd

# Create sparse time series with gaps
dates = pd.DatetimeIndex(['2024-01-01', '2024-01-03', '2024-01-05', '2024-01-08'])
values = pd.Series([10, 20, 15, 25], index=dates)

# Forward fill missing values
filled_forward = values.resample('D').ffill()
print("Forward fill:")
print(filled_forward)

# Interpolate missing values
interpolated = values.resample('D').interpolate()
print("\nInterpolated:")
print(interpolated)

Forward fill:
2024-01-01    10
2024-01-02    10
2024-01-03    20
2024-01-04    20
2024-01-05    15
2024-01-06    15
2024-01-07    15
2024-01-08    25
Freq: D, dtype: int64

Interpolated:
2024-01-01    10.000000
2024-01-02    15.000000
2024-01-03    20.000000
2024-01-04    17.500000
2024-01-05    15.000000
2024-01-06    18.333333
2024-01-07    21.666667
2024-01-08    25.000000
Freq: D, dtype: float64


---

## Rolling Windows

Rolling windows compute statistics over a moving window of data. This is essential for smoothing noise and identifying trends in time series.

### Fixed-Size Rolling Windows

In [18]:
import pandas as pd
import numpy as np

# Create sample data with noise
dates = pd.date_range('2024-01-01', periods=50, freq='D')
signal = 100 + 10 * np.sin(np.arange(50) * 2 * np.pi / 50)
noise = np.random.normal(0, 2, 50)
ts = pd.Series(signal + noise, index=dates)

# 7-day rolling average
rolling_mean = ts.rolling(window=7).mean()

# 7-day rolling standard deviation
rolling_std = ts.rolling(window=7).std()

print(pd.DataFrame({
    'original': ts,
    'rolling_mean': rolling_mean,
    'rolling_std': rolling_std
}).head(15))

              original  rolling_mean  rolling_std
2024-01-01  100.985566           NaN          NaN
2024-01-02   99.224964           NaN          NaN
2024-01-03   99.372228           NaN          NaN
2024-01-04  102.734530           NaN          NaN
2024-01-05  106.016133           NaN          NaN
2024-01-06  106.088175           NaN          NaN
2024-01-07  105.788596    102.887170     3.105401
2024-01-08  107.837722    103.866050     3.465196
2024-01-09  108.645037    105.211774     3.179825
2024-01-10  107.786229    106.413775     1.961361
2024-01-11  109.382921    107.363545     1.416931
2024-01-12  109.430018    107.851242     1.462646
2024-01-13  107.438379    108.044129     1.267387
2024-01-14  111.002740    108.789006     1.252974
2024-01-15  107.185784    108.695872     1.355503


**Intuition:** The first 6 values of a 7-day rolling mean are NaN—you need 7 data points to compute a 7-day average.

### The `min_periods` Parameter

Control how many values are required before a result is calculated:

In [19]:
import pandas as pd

data = pd.Series(
    [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    index=pd.date_range('2024-01-01', periods=10, freq='D')
)

# Default: requires full window (6 NaNs at start)
rolling_default = data.rolling(window=7).mean()

# With min_periods=3: calculate when 3 values are available (only 2 NaNs)
rolling_flexible = data.rolling(window=7, min_periods=3).mean()

print(pd.DataFrame({
    'data': data,
    'default': rolling_default,
    'min_periods=3': rolling_flexible
}))

            data  default  min_periods=3
2024-01-01     1      NaN            NaN
2024-01-02     2      NaN            NaN
2024-01-03     3      NaN            2.0
2024-01-04     4      NaN            2.5
2024-01-05     5      NaN            3.0
2024-01-06     6      NaN            3.5
2024-01-07     7      4.0            4.0
2024-01-08     8      5.0            5.0
2024-01-09     9      6.0            6.0
2024-01-10    10      7.0            7.0


### Centered Windows

By default, the window is right-aligned (trailing). Use `center=True` to center the window:

In [20]:
import pandas as pd

data = pd.Series(
    [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    index=pd.date_range('2024-01-01', periods=10, freq='D')
)

trailing = data.rolling(window=3).mean()
centered = data.rolling(window=3, center=True).mean()

print(pd.DataFrame({'data': data, 'trailing': trailing, 'centered': centered}))

            data  trailing  centered
2024-01-01     1       NaN       NaN
2024-01-02     2       NaN       2.0
2024-01-03     3       2.0       3.0
2024-01-04     4       3.0       4.0
2024-01-05     5       4.0       5.0
2024-01-06     6       5.0       6.0
2024-01-07     7       6.0       7.0
2024-01-08     8       7.0       8.0
2024-01-09     9       8.0       9.0
2024-01-10    10       9.0       NaN


**Use case:** Centered windows are useful for smoothing historical data while preserving timing, but they require future data and are not suitable for real-time analysis.

### Time-Based Rolling Windows

Instead of a fixed number of observations, you can use a time period. This is particularly useful when your data has irregular spacing:

In [21]:
import pandas as pd
import numpy as np

# Create irregular time series
times = pd.to_datetime([
    '2024-01-01 09:00:00',
    '2024-01-01 09:00:02',
    '2024-01-01 09:00:03',
    '2024-01-01 09:00:05',
    '2024-01-01 09:00:06'
])

values = pd.Series([0.0, 1.0, 2.0, np.nan, 4.0], index=times)

print("Original data:")
print(values)

# 2-second rolling window
rolling_2s = values.rolling('2s').sum()
print("\n2-second rolling sum:")
print(rolling_2s)

# 3-second rolling window
rolling_3s = values.rolling('3s').mean()
print("\n3-second rolling mean:")
print(rolling_3s)

Original data:
2024-01-01 09:00:00    0.0
2024-01-01 09:00:02    1.0
2024-01-01 09:00:03    2.0
2024-01-01 09:00:05    NaN
2024-01-01 09:00:06    4.0
dtype: float64

2-second rolling sum:
2024-01-01 09:00:00    0.0
2024-01-01 09:00:02    1.0
2024-01-01 09:00:03    3.0
2024-01-01 09:00:05    NaN
2024-01-01 09:00:06    4.0
dtype: float64

3-second rolling mean:
2024-01-01 09:00:00    0.0
2024-01-01 09:00:02    0.5
2024-01-01 09:00:03    1.5
2024-01-01 09:00:05    2.0
2024-01-01 09:00:06    4.0
dtype: float64


### Advanced Rolling Operations

In [22]:
import pandas as pd
import numpy as np

np.random.seed(42)
dates = pd.date_range('2024-01-01', periods=50, freq='D')
ts = pd.Series(100 + np.random.randn(50).cumsum(), index=dates)

# Multiple statistics at once
rolling_stats = ts.rolling(window=14).agg([
    ('mean', 'mean'),
    ('std', 'std'),
    ('min', 'min'),
    ('max', 'max')
])
print(rolling_stats.head(20))

# Custom rolling function
def coefficient_of_variation(x):
    return x.std() / x.mean()

cv = ts.rolling(window=7).apply(coefficient_of_variation)
print("\nCoefficient of variation (7-day):")
print(cv.head(10))

                  mean                   std                   min  \
                  mean                   std                   min   
                  mean        mean       std       std         min   
2024-01-01         NaN         NaN       NaN       NaN         NaN   
2024-01-02         NaN         NaN       NaN       NaN         NaN   
2024-01-03         NaN         NaN       NaN       NaN         NaN   
2024-01-04         NaN         NaN       NaN       NaN         NaN   
2024-01-05         NaN         NaN       NaN       NaN         NaN   
2024-01-06         NaN         NaN       NaN       NaN         NaN   
2024-01-07         NaN         NaN       NaN       NaN         NaN   
2024-01-08         NaN         NaN       NaN       NaN         NaN   
2024-01-09         NaN         NaN       NaN       NaN         NaN   
2024-01-10         NaN         NaN       NaN       NaN         NaN   
2024-01-11         NaN         NaN       NaN       NaN         NaN   
2024-01-12         N

### Expanding Windows

An **expanding window** grows over time, including all data from the start. Use it for cumulative statistics:

In [23]:
import pandas as pd

df = pd.DataFrame({
    'price': [10, 12, 11, 15, 14, 16, 13, 17, 18, 19]
})

df['cumulative_avg'] = df['price'].expanding().mean()
df['cumulative_max'] = df['price'].expanding().max()
df['cumulative_sum'] = df['price'].expanding().sum()

print(df)

   price  cumulative_avg  cumulative_max  cumulative_sum
0     10            10.0            10.0            10.0
1     12            11.0            12.0            22.0
2     11            11.0            12.0            33.0
3     15            12.0            15.0            48.0
4     14            12.4            15.0            62.0
5     16            13.0            16.0            78.0
6     13            13.0            16.0            91.0
7     17            13.5            17.0           108.0
8     18            14.0            18.0           126.0
9     19            14.5            19.0           145.0


### Exponentially Weighted Windows (EWM)

Exponentially weighted windows give more weight to recent observations. They react faster to changes than simple moving averages and produce no NaN values at the start:

In [24]:
import pandas as pd
import numpy as np

np.random.seed(42)
dates = pd.date_range('2023-01-01', periods=100, freq='D')
prices = 100 + np.cumsum(np.random.randn(100))
df = pd.DataFrame({'price': prices}, index=dates)

df['MA_7'] = df['price'].rolling(window=7).mean()
df['ewm_14'] = df['price'].ewm(span=14).mean()
df['ewm_30'] = df['price'].ewm(span=30).mean()

print(df.head(15))

                 price        MA_7      ewm_14      ewm_30
2023-01-01  100.496714         NaN  100.496714  100.496714
2023-01-02  100.358450         NaN  100.422644  100.425278
2023-01-03  101.006138         NaN  100.645541  100.631944
2023-01-04  102.529168         NaN  101.221796  101.154699
2023-01-05  102.295015         NaN  101.501797  101.414148
2023-01-06  102.060878         NaN  101.631158  101.540671
2023-01-07  103.640091  101.769493  102.054483  101.903779
2023-01-08  104.407525  102.328181  102.514703  102.294453
2023-01-09  103.938051  102.839552  102.776775  102.529411
2023-01-10  104.480611  103.335906  103.075327  102.788053
2023-01-11  104.017193  103.548481  103.233729  102.940603
2023-01-12  103.551464  103.727973  103.285366  103.012154
2023-01-13  103.793426  103.975480  103.365592  103.099091
2023-01-14  101.880146  103.724059  103.136655  102.969510
2023-01-15  100.155228  103.116588  102.686514  102.682338


**When to use each method:**

| Method | Responsiveness | Best For |
|--------|---------------|----------|
| Rolling (SMA) | Moderate | Stable trends, equal weight |
| EWM | High | Recent data more important |
| Expanding | Low | Cumulative statistics |

### Rolling with Multiple Columns

In [25]:
import pandas as pd

df = pd.DataFrame({
    'sales': [100, 105, 110, 108, 115, 120, 118, 125],
    'costs': [60, 62, 65, 64, 68, 70, 69, 72]
}, index=pd.date_range('2024-01-01', periods=8, freq='D'))

# 3-day rolling mean for all columns
print("3-day rolling mean:")
print(df.rolling(window=3).mean())

# Different aggregations per column
print("\n3-day rolling (mean for sales, max for costs):")
print(df.rolling(window=3).agg({'sales': 'mean', 'costs': 'max'}))

3-day rolling mean:
                 sales      costs
2024-01-01         NaN        NaN
2024-01-02         NaN        NaN
2024-01-03  105.000000  62.333333
2024-01-04  107.666667  63.666667
2024-01-05  111.000000  65.666667
2024-01-06  114.333333  67.333333
2024-01-07  117.666667  69.000000
2024-01-08  121.000000  70.333333

3-day rolling (mean for sales, max for costs):
                 sales  costs
2024-01-01         NaN    NaN
2024-01-02         NaN    NaN
2024-01-03  105.000000   65.0
2024-01-04  107.666667   65.0
2024-01-05  111.000000   68.0
2024-01-06  114.333333   70.0
2024-01-07  117.666667   70.0
2024-01-08  121.000000   72.0


---

## Date Offsets

Date offsets allow you to shift dates by specific periods or align them to particular boundaries. They understand business days, month boundaries, and other calendar rules.

### Common Offset Operations

In [26]:
import pandas as pd
from pandas.tseries.offsets import Day, Hour, BDay, Week, MonthEnd, QuarterEnd

start_date = pd.Timestamp('2024-01-15')

print(f"Start: {start_date}")
print(f"Add 5 days: {start_date + Day(5)}")
print(f"Add 3 hours: {start_date + Hour(3)}")
print(f"Add 5 business days: {start_date + BDay(5)}")
print(f"Add 2 weeks: {start_date + Week(2)}")
print(f"Move to month end: {start_date + MonthEnd()}")
print(f"Move to quarter end: {start_date + QuarterEnd()}")

Start: 2024-01-15 00:00:00
Add 5 days: 2024-01-20 00:00:00
Add 3 hours: 2024-01-15 03:00:00
Add 5 business days: 2024-01-22 00:00:00
Add 2 weeks: 2024-01-29 00:00:00
Move to month end: 2024-01-31 00:00:00
Move to quarter end: 2024-03-31 00:00:00


### Business Day vs Calendar Day

In [27]:
import pandas as pd
from pandas.tseries.offsets import Day, BDay

date = pd.Timestamp('2024-01-19')  # Friday
print(f"Start date: {date} ({date.strftime('%A')})")
print(f"Adding 5 calendar days: {date + 5 * Day()}")
print(f"Adding 5 business days: {date + 5 * BDay()}")
# Business day calculation skips the weekend

Start date: 2024-01-19 00:00:00 (Friday)
Adding 5 calendar days: 2024-01-24 00:00:00
Adding 5 business days: 2024-01-26 00:00:00


### Generating Sequences with Offsets

In [28]:
import pandas as pd
from pandas.tseries.offsets import BMonthEnd

# Generate business day sequence
business_days = pd.bdate_range('2024-01-01', periods=10)
print("Business days:")
print(business_days)

# Generate month-end dates
month_ends = pd.date_range('2024-01-01', periods=12, freq='ME')
print("\nMonth ends:")
print(month_ends)

# Generate business month-end dates
custom_dates = pd.date_range('2024-01-01', periods=6, freq=BMonthEnd())
print("\nBusiness month ends:")
print(custom_dates)

Business days:
DatetimeIndex(['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04',
               '2024-01-05', '2024-01-08', '2024-01-09', '2024-01-10',
               '2024-01-11', '2024-01-12'],
              dtype='datetime64[ns]', freq='B')

Month ends:
DatetimeIndex(['2024-01-31', '2024-02-29', '2024-03-31', '2024-04-30',
               '2024-05-31', '2024-06-30', '2024-07-31', '2024-08-31',
               '2024-09-30', '2024-10-31', '2024-11-30', '2024-12-31'],
              dtype='datetime64[ns]', freq='ME')

Business month ends:
DatetimeIndex(['2024-01-31', '2024-02-29', '2024-03-29', '2024-04-30',
               '2024-05-31', '2024-06-28'],
              dtype='datetime64[ns]', freq='BME')


### Rolling Dates Forward and Backward

In [29]:
import pandas as pd
from pandas.tseries.offsets import MonthEnd

test_date = pd.Timestamp('2024-01-15')
month_end = MonthEnd()

print(f"Is {test_date} on month end? {month_end.is_on_offset(test_date)}")

rolled_forward = month_end.rollforward(test_date)
print(f"Roll forward to: {rolled_forward}")

rolled_backward = month_end.rollback(test_date)
print(f"Roll backward to: {rolled_backward}")

Is 2024-01-15 00:00:00 on month end? False
Roll forward to: 2024-01-31 00:00:00
Roll backward to: 2023-12-31 00:00:00


### Custom Business Day Calendars

For financial analysis, you often need to exclude holidays:

In [30]:
from pandas.tseries.offsets import CustomBusinessDay
import pandas as pd

# US market holidays in 2024
us_holidays = [
    '2024-01-01',  # New Year
    '2024-01-15',  # MLK Day
    '2024-05-27',  # Memorial Day
    '2024-07-04',  # Independence Day
    '2024-09-02',  # Labor Day
    '2024-11-28',  # Thanksgiving
    '2024-12-25',  # Christmas
]

us_bday = CustomBusinessDay(holidays=us_holidays)

trades = pd.DataFrame({
    'trade_date': pd.to_datetime(['2024-12-23', '2024-12-24', '2024-12-26']),
    'amount': [100000, 250000, 150000]
})

# Settlement is T+2 business days
trades['settlement_date'] = trades['trade_date'] + 2 * us_bday
print(trades)

  trade_date  amount settlement_date
0 2024-12-23  100000      2024-12-26
1 2024-12-24  250000      2024-12-27
2 2024-12-26  150000      2024-12-30


/var/folders/6k/wdwghqdd7ynf_52q9qqp3v9w0000gp/T/ipykernel_38883/2274535090.py:23: PerformanceWarning: Non-vectorized DateOffset being applied to Series or DatetimeIndex.
  trades['settlement_date'] = trades['trade_date'] + 2 * us_bday


---

## Time Zones

Working with time zones is essential when dealing with global data. Pandas distinguishes between **naive** (no timezone info) and **aware** (timezone-aware) datetimes.

### Naive vs. Aware Datetimes

In [31]:
import pandas as pd

# Naive datetime (no timezone info)
naive_ts = pd.Timestamp('2023-01-15 12:00:00')
print(f"Naive: {naive_ts}")
print(f"Timezone: {naive_ts.tz}")  # None

# Aware datetime (with timezone)
aware_ts = pd.Timestamp('2023-01-15 12:00:00', tz='UTC')
print(f"Aware: {aware_ts}")
print(f"Timezone: {aware_ts.tz}")  # UTC

Naive: 2023-01-15 12:00:00
Timezone: None
Aware: 2023-01-15 12:00:00+00:00
Timezone: UTC


**Key insight:** The actual moment in time is the same across timezones; only the representation changes. 12:00 UTC = 07:00 Eastern = 21:00 Tokyo.

### Localization: Adding Timezone Info

Use `.tz_localize()` to attach timezone information to naive datetimes:

In [32]:
import pandas as pd

# Create naive datetime index
dates = pd.date_range('2024-01-01', periods=3, freq='D')
print(f"Timezone before: {dates.tz}")  # None

# Localize to UTC
dates_utc = dates.tz_localize('UTC')
print(dates_utc)

# Localize to a specific timezone
dates_eastern = dates.tz_localize('US/Eastern')
print(dates_eastern)

Timezone before: None
DatetimeIndex(['2024-01-01 00:00:00+00:00', '2024-01-02 00:00:00+00:00',
               '2024-01-03 00:00:00+00:00'],
              dtype='datetime64[ns, UTC]', freq='D')
DatetimeIndex(['2024-01-01 00:00:00-05:00', '2024-01-02 00:00:00-05:00',
               '2024-01-03 00:00:00-05:00'],
              dtype='datetime64[ns, US/Eastern]', freq=None)


### Conversion: Changing Timezones

Use `.tz_convert()` to convert between timezones:

In [ ]:
import pandas as pd

dates_utc = pd.date_range('2023-01-15 12:00:00', periods=3, freq='h', tz='UTC')
print("UTC:")
print(dates_utc)

dates_eastern = dates_utc.tz_convert('US/Eastern')
print("\nEastern Time:")
print(dates_eastern)

dates_tokyo = dates_utc.tz_convert('Asia/Tokyo')
print("\nTokyo Time:")
print(dates_tokyo)

### Handling Daylight Saving Time

DST transitions create times that either don't exist (spring forward) or are ambiguous (fall back):

In [34]:
import pandas as pd

# Spring forward: 2:30 AM doesn't exist on March 10, 2024
try:
    pd.to_datetime('2024-03-10 02:30:00').tz_localize('US/Eastern')
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")

# Solutions using the nonexistent parameter
result_fwd = pd.to_datetime('2024-03-10 02:30:00').tz_localize(
    'US/Eastern', nonexistent='shift_forward')
print(f"Shift forward: {result_fwd}")

result_nat = pd.to_datetime('2024-03-10 02:30:00').tz_localize(
    'US/Eastern', nonexistent='NaT')
print(f"Set to NaT: {result_nat}")

# Fall back: 1:30 AM is ambiguous on November 3, 2024
result_dst = pd.to_datetime('2024-11-03 01:30:00').tz_localize(
    'US/Eastern', ambiguous=True)   # First occurrence (DST)
print(f"First occurrence (DST): {result_dst}")

result_std = pd.to_datetime('2024-11-03 01:30:00').tz_localize(
    'US/Eastern', ambiguous=False)  # Second occurrence (Standard)
print(f"Second occurrence (Standard): {result_std}")

Error: NonExistentTimeError: 2024-03-10 02:30:00
Shift forward: 2024-03-10 03:00:00-04:00
Set to NaT: NaT
First occurrence (DST): 2024-11-03 01:30:00-04:00
Second occurrence (Standard): 2024-11-03 01:30:00-05:00


### Common Pitfalls with Time Zones

In [35]:
import pandas as pd

# Pitfall 1: Can't convert a naive datetime directly
naive_date = pd.to_datetime('2024-01-15 10:00:00')
# naive_date.tz_convert('US/Eastern')  # AttributeError!

# Fix: Localize first, then convert
aware_date = naive_date.tz_localize('UTC')
result = aware_date.tz_convert('US/Eastern')
print(f"Correct result: {result}")

# Pitfall 2: Can't mix naive and aware datetimes
naive = pd.to_datetime('2024-01-15')
aware = pd.to_datetime('2024-01-16').tz_localize('UTC')
# naive + aware  # TypeError!

# Fix: Make both aware
naive_aware = naive.tz_localize('UTC')
result = aware - naive_aware
print(f"Timedelta: {result}")

Correct result: 2024-01-15 05:00:00-05:00
Timedelta: 1 days 00:00:00


---

## Handling Missing Data in Time Series

Time series often have gaps. Different strategies work for different situations.

### Forward Fill, Backward Fill, and Interpolation

In [36]:
import pandas as pd
import numpy as np

dates = pd.date_range('2023-01-01', periods=10, freq='D')
values = [100, 102, np.nan, np.nan, 105, 107, np.nan, 110, 112, 115]
ts = pd.Series(values, index=dates)

print("Original data:")
print(ts)

# Forward fill: carry last value forward
print("\nForward fill:")
print(ts.ffill())

# Backward fill: use next value
print("\nBackward fill:")
print(ts.bfill())

# Limit consecutive fills
print("\nForward fill (limit=1):")
print(ts.ffill(limit=1))

# Linear interpolation
print("\nLinear interpolation:")
print(ts.interpolate(method='linear'))

Original data:
2023-01-01    100.0
2023-01-02    102.0
2023-01-03      NaN
2023-01-04      NaN
2023-01-05    105.0
2023-01-06    107.0
2023-01-07      NaN
2023-01-08    110.0
2023-01-09    112.0
2023-01-10    115.0
Freq: D, dtype: float64

Forward fill:
2023-01-01    100.0
2023-01-02    102.0
2023-01-03    102.0
2023-01-04    102.0
2023-01-05    105.0
2023-01-06    107.0
2023-01-07    107.0
2023-01-08    110.0
2023-01-09    112.0
2023-01-10    115.0
Freq: D, dtype: float64

Backward fill:
2023-01-01    100.0
2023-01-02    102.0
2023-01-03    105.0
2023-01-04    105.0
2023-01-05    105.0
2023-01-06    107.0
2023-01-07    110.0
2023-01-08    110.0
2023-01-09    112.0
2023-01-10    115.0
Freq: D, dtype: float64

Forward fill (limit=1):
2023-01-01    100.0
2023-01-02    102.0
2023-01-03    102.0
2023-01-04      NaN
2023-01-05    105.0
2023-01-06    107.0
2023-01-07    107.0
2023-01-08    110.0
2023-01-09    112.0
2023-01-10    115.0
Freq: D, dtype: float64

Linear interpolation:
2023-01-01

**When to use each:**
- **Forward fill:** Stock prices, inventory levels (assume value stays constant until updated)
- **Backward fill:** Less common; useful when you know future values
- **Linear interpolation:** Continuous variables with smooth trends (temperatures, prices)

### Detecting and Handling Irregular Time Series

In [37]:
import pandas as pd

# Create irregular time series (gaps in data)
irregular_dates = pd.DatetimeIndex([
    '2023-01-01', '2023-01-02', '2023-01-05',  # Gap: Jan 3-4
    '2023-01-06', '2023-01-10'                  # Gap: Jan 7-9
])
irregular_data = pd.Series([100, 102, 110, 112, 120], index=irregular_dates)

# Detect gaps
time_diffs = irregular_data.index.to_series().diff()
gaps = time_diffs[time_diffs > pd.Timedelta('1D')]
print("Gaps larger than 1 day:")
print(gaps)

# Resample to regular frequency (creates NaN for missing dates)
regular_data = irregular_data.asfreq('D')
print("\nAfter resampling to daily:")
print(regular_data)

print("\nLinear interpolation:")
print(regular_data.interpolate(method='linear'))

Gaps larger than 1 day:
2023-01-05   3 days
2023-01-10   4 days
dtype: timedelta64[ns]

After resampling to daily:
2023-01-01    100.0
2023-01-02    102.0
2023-01-03      NaN
2023-01-04      NaN
2023-01-05    110.0
2023-01-06    112.0
2023-01-07      NaN
2023-01-08      NaN
2023-01-09      NaN
2023-01-10    120.0
Freq: D, dtype: float64

Linear interpolation:
2023-01-01    100.000000
2023-01-02    102.000000
2023-01-03    104.666667
2023-01-04    107.333333
2023-01-05    110.000000
2023-01-06    112.000000
2023-01-07    114.000000
2023-01-08    116.000000
2023-01-09    118.000000
2023-01-10    120.000000
Freq: D, dtype: float64


---

## Shift and Diff: Comparing Values Across Time

In [38]:
import pandas as pd

prices = pd.Series(
    [100, 102, 101, 105, 103],
    index=pd.date_range('2023-01-01', periods=5, freq='D')
)

# shift(1): look at previous values (lag)
print("Previous day's price:")
print(prices.shift(1))

# shift(-1): look at future values (lead)
print("\nTomorrow's price:")
print(prices.shift(-1))

# diff(): calculate change between consecutive periods
print("\nDaily price change:")
print(prices.diff())

# Percentage change
print("\nDaily percentage change:")
print(prices.pct_change() * 100)

Previous day's price:
2023-01-01      NaN
2023-01-02    100.0
2023-01-03    102.0
2023-01-04    101.0
2023-01-05    105.0
Freq: D, dtype: float64

Tomorrow's price:
2023-01-01    102.0
2023-01-02    101.0
2023-01-03    105.0
2023-01-04    103.0
2023-01-05      NaN
Freq: D, dtype: float64

Daily price change:
2023-01-01    NaN
2023-01-02    2.0
2023-01-03   -1.0
2023-01-04    4.0
2023-01-05   -2.0
Freq: D, dtype: float64

Daily percentage change:
2023-01-01         NaN
2023-01-02    2.000000
2023-01-03   -0.980392
2023-01-04    3.960396
2023-01-05   -1.904762
Freq: D, dtype: float64


---

## Practical Example: Stock Price Analysis

Let's combine these techniques to analyze a realistic scenario:

In [39]:
import pandas as pd
import numpy as np

np.random.seed(42)
dates = pd.date_range('2023-01-01', periods=252, freq='B')  # 252 business days
prices = 100 + np.cumsum(np.random.randn(252) * 2)
volume = np.random.randint(1000000, 5000000, 252)

stock_df = pd.DataFrame({
    'close': prices,
    'volume': volume
}, index=dates)

# Calculate technical indicators
stock_df['MA_20'] = stock_df['close'].rolling(20).mean()
stock_df['MA_50'] = stock_df['close'].rolling(50).mean()
stock_df['daily_return'] = stock_df['close'].pct_change()
stock_df['volatility'] = stock_df['daily_return'].rolling(20).std()

print("Stock data sample:")
print(stock_df.head())

# Resample to weekly data
weekly_data = stock_df.resample('W').agg({
    'close': 'last',
    'volume': 'sum'
})
print("\nWeekly data (first 5 rows):")
print(weekly_data.head())

# Resample to monthly with multiple aggregations
monthly_data = stock_df.resample('ME').agg({
    'close': ['first', 'last', 'min', 'max'],
    'volume': 'sum',
    'volatility': 'mean'
})
print("\nMonthly summary (first 3 rows):")
print(monthly_data.head(3))

# Identify best and worst trading days
best_idx = stock_df['daily_return'].idxmax()
worst_idx = stock_df['daily_return'].idxmin()
print(f"\nBest trading day: {best_idx.date()} ({stock_df['daily_return'].max():.2%})")
print(f"Worst trading day: {worst_idx.date()} ({stock_df['daily_return'].min():.2%})")

Stock data sample:
                 close   volume  MA_20  MA_50  daily_return  volatility
2023-01-02  100.993428  4280143    NaN    NaN           NaN         NaN
2023-01-03  100.716900  1406716    NaN    NaN     -0.002738         NaN
2023-01-04  102.012277  3543590    NaN    NaN      0.012862         NaN
2023-01-05  105.058336  4699391    NaN    NaN      0.029860         NaN
2023-01-06  104.590030  3762174    NaN    NaN     -0.004458         NaN

Weekly data (first 5 rows):
                 close    volume
2023-01-08  104.590030  17692014
2023-01-15  108.961222  16002221
2023-01-22  100.310456  14659657
2023-01-29   93.148058  12268561
2023-02-05   91.824597  13958076

Monthly summary (first 3 rows):
                 close                                      volume volatility
                 first       last        min         max       sum       mean
2023-01-31  100.993428  95.627802  93.148058  108.961222  64818023   0.020033
2023-02-28   95.762859  84.328723  82.115330   95.76285

---

## Common Pitfalls and How to Avoid Them

### Pitfall 1: Resampling Without a DatetimeIndex

In [40]:
import pandas as pd

df = pd.DataFrame({
    'date': ['2023-01-01', '2023-01-02', '2023-01-03'],
    'value': [100, 102, 105]
})

# df.resample('W').sum()  # Error: cannot resample on non-DatetimeIndex

# Fix: Set DatetimeIndex first
df['date'] = pd.to_datetime(df['date'])
df = df.set_index('date')
result = df.resample('W').sum()
print(result)

            value
date             
2023-01-01    100
2023-01-08    207


### Pitfall 2: Confusing `shift()` and `diff()`

In [41]:
import pandas as pd

prices = pd.Series([100, 102, 101, 105],
                   index=pd.date_range('2023-01-01', periods=4, freq='D'))

print("shift(1) - Moves data forward (creates lag):")
print(prices.shift(1))

print("\ndiff() - Calculates differences:")
print(prices.diff())

shift(1) - Moves data forward (creates lag):
2023-01-01      NaN
2023-01-02    100.0
2023-01-03    102.0
2023-01-04    101.0
Freq: D, dtype: float64

diff() - Calculates differences:
2023-01-01    NaN
2023-01-02    2.0
2023-01-03   -1.0
2023-01-04    4.0
Freq: D, dtype: float64


### Pitfall 3: Not Handling NaN Values After Rolling

In [42]:
import pandas as pd
import numpy as np

prices = pd.Series([100, 102, 101, 105],
                   index=pd.date_range('2023-01-01', periods=4, freq='D'))

returns = prices.pct_change()

# Be explicit about NaN handling
print(returns.dropna().mean())   # Drop NaNs
print(returns.fillna(0).mean())  # Fill with 0 if appropriate

0.016600012942470748
0.01245000970685306


---

## Summary: When to Use Each Method

| Task | Method | Example |
|------|--------|---------|
| Convert string to datetime | `pd.to_datetime()` | `pd.to_datetime('2023-01-01')` |
| Create date range | `pd.date_range()` | `pd.date_range('2023-01-01', periods=10)` |
| Reduce frequency | `resample().agg()` | `data.resample('W').sum()` |
| Increase frequency | `resample().ffill()` | `data.resample('D').ffill()` |
| Moving average | `rolling().mean()` | `data.rolling(7).mean()` |
| Cumulative sum | `expanding().sum()` | `data.expanding().sum()` |
| Weighted moving average | `ewm().mean()` | `data.ewm(span=14).mean()` |
| Fill gaps | `ffill()` or `interpolate()` | `data.ffill()` |
| Get previous value | `shift()` | `data.shift(1)` |
| Calculate change | `diff()` | `data.diff()` |
| Add timezone | `tz_localize()` | `dates.tz_localize('UTC')` |
| Change timezone | `tz_convert()` | `dates.tz_convert('US/Eastern')` |
| Business day arithmetic | `BDay()` offset | `date + BDay(5)` |
| Month-end alignment | `MonthEnd()` offset | `date + MonthEnd()` |

**Key takeaways:**
- **Parse dates early** during import for performance and correctness
- **Set a DatetimeIndex** before resampling or time-based slicing
- **Use `.dt` accessor** to extract datetime components from a Series column
- **Resample** to change data frequency; **roll** to smooth within the same frequency
- **Handle timezones** with `tz_localize()` (add) and `tz_convert()` (change)
- **Use offsets** for business-calendar-aware date arithmetic
- **Handle NaN values explicitly** after rolling, shifting, or upsampling

---

# Exercises

Test your understanding of this chapter's concepts.

### Exercise 1: Creating and Indexing a DateTime Series

Practice creating a pandas DataFrame with a DatetimeIndex and performing basic datetime-based indexing. Create a daily sales DataFrame for January 2024 and select specific date ranges using datetime indexing.

In [43]:
import pandas as pd
import numpy as np

# TODO: Create a date range for all days in January 2024 with daily frequency
# Hint: use pd.date_range() with start, end (or periods), and freq arguments
dates = None

# TODO: Create a DataFrame with the dates as the index and a 'sales' column
# Use this sales data: [150, 200, 180, 220, 195, 210, 175, 230, 205, 190,
#                       215, 225, 240, 185, 200, 195, 210, 220, 230, 215,
#                       205, 195, 225, 240, 210, 200, 215, 195, 220, 230, 210]
df = None

# TODO: Print the first 5 rows of the DataFrame

# TODO: Select only the sales data from January 8 to January 14, 2024
week2 = None
print(week2)

# TODO: Print the data type of the index to confirm it is a DatetimeIndex


None


### Exercise 2: Resampling Time Series Data

Use pandas resampling to aggregate a daily temperature dataset into weekly and monthly summaries. Practice using different aggregation functions (mean, max, min) with the resample method.

In [44]:
import pandas as pd
import numpy as np

# Sample daily temperature data for 3 months (Jan-Mar 2024)
np.random.seed(42)
dates = pd.date_range(start='2024-01-01', end='2024-03-31', freq='D')
temperatures = np.random.normal(loc=15, scale=8, size=len(dates)).round(1)
df = pd.DataFrame({'temperature': temperatures}, index=dates)
print("Original daily data (first 10 rows):")
print(df.head(10))

# TODO: Resample the data to weekly frequency and calculate the mean temperature
# Hint: use df.resample('W') and .mean()
weekly_mean = None
print("\nWeekly mean temperature:")
print(weekly_mean)

# TODO: Resample the data to monthly frequency and calculate:
#   - monthly_max: the maximum temperature each month
#   - monthly_min: the minimum temperature each month
#   - monthly_mean: the average temperature each month
monthly_max = None
monthly_min = None
monthly_mean = None

# TODO: Combine monthly_max, monthly_min, and monthly_mean into one DataFrame
# with columns named 'max', 'min', and 'mean'
monthly_summary = None
print("\nMonthly temperature summary:")
print(monthly_summary)


Original daily data (first 10 rows):
            temperature
2024-01-01         19.0
2024-01-02         13.9
2024-01-03         20.2
2024-01-04         27.2
2024-01-05         13.1
2024-01-06         13.1
2024-01-07         27.6
2024-01-08         21.1
2024-01-09         11.2
2024-01-10         19.3

Weekly mean temperature:
None

Monthly temperature summary:
None


### Exercise 3: Rolling Windows and Shift Operations

Apply rolling window calculations and shift operations to analyze a stock price dataset. Calculate a 7-day rolling average, compute daily returns using shift, and identify days where the price increased compared to the previous day.

In [45]:
import pandas as pd
import numpy as np

# Sample stock price data for 30 trading days
np.random.seed(7)
dates = pd.date_range(start='2024-01-02', periods=30, freq='B')  # 'B' = business days
prices = (100 + np.cumsum(np.random.normal(0, 2, 30))).round(2)
df = pd.DataFrame({'price': prices}, index=dates)
print("Stock prices (first 10 rows):")
print(df.head(10))

# TODO: Calculate a 7-day rolling mean of the price and add it as a new column '7d_avg'
# Hint: use df['price'].rolling(window=7).mean()
df['7d_avg'] = None

# TODO: Calculate the daily return as the percentage change from the previous day
# Formula: (today's price - yesterday's price) / yesterday's price * 100
# Hint: use .shift(1) to get the previous day's price
df['daily_return_pct'] = None

# TODO: Create a boolean column 'price_increased' that is True when
# the price is higher than the previous day's price
df['price_increased'] = None

# TODO: Count how many days the price increased vs decreased
# Hint: use .value_counts() on the 'price_increased' column
print("\nPrice increase/decrease count:")
# your code here

print("\nFull DataFrame (last 10 rows):")
print(df.tail(10).round(2))


Stock prices (first 10 rows):
             price
2024-01-02  103.38
2024-01-03  102.45
2024-01-04  102.51
2024-01-05  103.33
2024-01-08  101.75
2024-01-09  101.76
2024-01-10  101.75
2024-01-11   98.24
2024-01-12  100.28
2024-01-15  101.48

Price increase/decrease count:

Full DataFrame (last 10 rows):
             price 7d_avg daily_return_pct price_increased
2024-01-30   99.14   None             None            None
2024-01-31   99.45   None             None            None
2024-02-01   98.67   None             None            None
2024-02-02  102.73   None             None            None
2024-02-05  102.64   None             None            None
2024-02-06   99.74   None             None            None
2024-02-07   98.93   None             None            None
2024-02-08   94.35   None             None            None
2024-02-09   96.45   None             None            None
2024-02-12   95.62   None             None            None


### Exercise 4: Handling Missing Data and Time Zones in Time Series

Work with a time series that has missing values and timezone information. Fill in gaps using forward-fill and interpolation methods, then convert timestamps between time zones to compare sensor readings from two different cities.

In [46]:
import pandas as pd
import numpy as np

# Sensor readings with some missing values (NaN)
dates = pd.date_range(start='2024-06-01 00:00', periods=12, freq='h',
                      tz='America/New_York')
readings = [23.1, np.nan, 24.5, np.nan, np.nan, 26.0,
            25.8, np.nan, 24.9, 23.7, np.nan, 22.5]
df = pd.DataFrame({'sensor_reading': readings}, index=dates)
print("Original sensor data with missing values:")
print(df)

# TODO: Count the number of missing values in the 'sensor_reading' column
print("\nNumber of missing values:", None)

# TODO: Create a new column 'ffill_reading' by filling missing values
# using forward fill (propagate the last valid observation forward)
df['ffill_reading'] = None

# TODO: Create a new column 'interp_reading' by filling missing values
# using linear interpolation
# Hint: use .interpolate(method='linear')
df['interp_reading'] = None

print("\nData after filling missing values:")
print(df)

# TODO: Convert the DatetimeIndex from 'America/New_York' to 'Europe/London'
# and store the result in a new DataFrame called df_london
# Hint: use .tz_convert()
df_london = None
print("\nSensor data in London time:")
print(df_london)

# TODO: Print the first timestamp in both New York and London time
# to verify the conversion is correct
print("\nFirst timestamp - New York:", None)
print("First timestamp - London: ", None)


Original sensor data with missing values:
                           sensor_reading
2024-06-01 00:00:00-04:00            23.1
2024-06-01 01:00:00-04:00             NaN
2024-06-01 02:00:00-04:00            24.5
2024-06-01 03:00:00-04:00             NaN
2024-06-01 04:00:00-04:00             NaN
2024-06-01 05:00:00-04:00            26.0
2024-06-01 06:00:00-04:00            25.8
2024-06-01 07:00:00-04:00             NaN
2024-06-01 08:00:00-04:00            24.9
2024-06-01 09:00:00-04:00            23.7
2024-06-01 10:00:00-04:00             NaN
2024-06-01 11:00:00-04:00            22.5

Number of missing values: None

Data after filling missing values:
                           sensor_reading ffill_reading interp_reading
2024-06-01 00:00:00-04:00            23.1          None           None
2024-06-01 01:00:00-04:00             NaN          None           None
2024-06-01 02:00:00-04:00            24.5          None           None
2024-06-01 03:00:00-04:00             NaN          None     

---

# Solutions

*Scroll down only after you've attempted the exercises above.*

<br><br><br><br><br><br><br><br><br><br>

### Solution 1: Creating and Indexing a DateTime Series

In [47]:
import pandas as pd
import numpy as np

# Create a date range for all days in January 2024 with daily frequency
dates = pd.date_range(start='2024-01-01', end='2024-01-31', freq='D')

# Create a DataFrame with the dates as the index and a 'sales' column
sales_data = [150, 200, 180, 220, 195, 210, 175, 230, 205, 190,
              215, 225, 240, 185, 200, 195, 210, 220, 230, 215,
              205, 195, 225, 240, 210, 200, 215, 195, 220, 230, 210]
df = pd.DataFrame({'sales': sales_data}, index=dates)

# Print the first 5 rows of the DataFrame
print(df.head())

# Select only the sales data from January 8 to January 14, 2024
week2 = df['2024-01-08':'2024-01-14']
print(week2)

# Print the data type of the index to confirm it is a DatetimeIndex
print(type(df.index))
print(df.index.dtype)


            sales
2024-01-01    150
2024-01-02    200
2024-01-03    180
2024-01-04    220
2024-01-05    195
            sales
2024-01-08    230
2024-01-09    205
2024-01-10    190
2024-01-11    215
2024-01-12    225
2024-01-13    240
2024-01-14    185
<class 'pandas.core.indexes.datetimes.DatetimeIndex'>
datetime64[ns]


### Solution 2: Resampling Time Series Data

In [48]:
import pandas as pd
import numpy as np

# Sample daily temperature data for 3 months (Jan-Mar 2024)
np.random.seed(42)
dates = pd.date_range(start='2024-01-01', end='2024-03-31', freq='D')
temperatures = np.random.normal(loc=15, scale=8, size=len(dates)).round(1)
df = pd.DataFrame({'temperature': temperatures}, index=dates)
print("Original daily data (first 10 rows):")
print(df.head(10))

# Resample the data to weekly frequency and calculate the mean temperature
weekly_mean = df.resample('W').mean()
print("\nWeekly mean temperature:")
print(weekly_mean)

# Resample the data to monthly frequency and calculate max, min, and mean
monthly_max = df.resample('ME').max()
monthly_min = df.resample('ME').min()
monthly_mean = df.resample('ME').mean()

# Combine into one DataFrame with columns named 'max', 'min', and 'mean'
monthly_summary = pd.DataFrame({
    'max': monthly_max['temperature'],
    'min': monthly_min['temperature'],
    'mean': monthly_mean['temperature'].round(2)
})
print("\nMonthly temperature summary:")
print(monthly_summary)


Original daily data (first 10 rows):
            temperature
2024-01-01         19.0
2024-01-02         13.9
2024-01-03         20.2
2024-01-04         27.2
2024-01-05         13.1
2024-01-06         13.1
2024-01-07         27.6
2024-01-08         21.1
2024-01-09         11.2
2024-01-10         19.3

Weekly mean temperature:
            temperature
2024-01-07    19.157143
2024-01-14    12.971429
2024-01-21    10.600000
2024-01-28    11.800000
2024-02-04    15.128571
2024-02-11    11.357143
2024-02-18    13.085714
2024-02-25    15.085714
2024-03-03    13.157143
2024-03-10    16.828571
2024-03-17    16.957143
2024-03-24    13.742857
2024-03-31    15.014286

Monthly temperature summary:
             max  min   mean
2024-01-31  27.6 -0.3  13.38
2024-02-29  29.8 -0.7  14.17
2024-03-31  27.5 -6.0  15.12


### Solution 3: Rolling Windows and Shift Operations

In [49]:
import pandas as pd
import numpy as np

# Sample stock price data for 30 trading days
np.random.seed(7)
dates = pd.date_range(start='2024-01-02', periods=30, freq='B')  # 'B' = business days
prices = (100 + np.cumsum(np.random.normal(0, 2, 30))).round(2)
df = pd.DataFrame({'price': prices}, index=dates)
print("Stock prices (first 10 rows):")
print(df.head(10))

# Calculate a 7-day rolling mean of the price and add it as a new column '7d_avg'
df['7d_avg'] = df['price'].rolling(window=7).mean()

# Calculate the daily return as the percentage change from the previous day
prev_price = df['price'].shift(1)
df['daily_return_pct'] = ((df['price'] - prev_price) / prev_price * 100).round(4)

# Create a boolean column 'price_increased' that is True when
# the price is higher than the previous day's price
df['price_increased'] = df['price'] > df['price'].shift(1)

# Count how many days the price increased vs decreased
print("\nPrice increase/decrease count:")
print(df['price_increased'].value_counts())

print("\nFull DataFrame (last 10 rows):")
print(df.tail(10).round(2))


Stock prices (first 10 rows):
             price
2024-01-02  103.38
2024-01-03  102.45
2024-01-04  102.51
2024-01-05  103.33
2024-01-08  101.75
2024-01-09  101.76
2024-01-10  101.75
2024-01-11   98.24
2024-01-12  100.28
2024-01-15  101.48

Price increase/decrease count:
price_increased
False    17
True     13
Name: count, dtype: int64

Full DataFrame (last 10 rows):
             price  7d_avg  daily_return_pct  price_increased
2024-01-30   99.14   98.17              3.44             True
2024-01-31   99.45   98.10              0.31             True
2024-02-01   98.67   98.35             -0.78            False
2024-02-02  102.73   99.01              4.11             True
2024-02-05  102.64   99.62             -0.09            False
2024-02-06   99.74   99.74             -2.83            False
2024-02-07   98.93  100.19             -0.81            False
2024-02-08   94.35   99.50             -4.63            False
2024-02-09   96.45   99.07              2.23             True
2024-02-12 

### Solution 4: Handling Missing Data and Time Zones in Time Series

In [50]:
import pandas as pd
import numpy as np

# Sensor readings with some missing values (NaN)
dates = pd.date_range(start='2024-06-01 00:00', periods=12, freq='h',
                      tz='America/New_York')
readings = [23.1, np.nan, 24.5, np.nan, np.nan, 26.0,
            25.8, np.nan, 24.9, 23.7, np.nan, 22.5]
df = pd.DataFrame({'sensor_reading': readings}, index=dates)
print("Original sensor data with missing values:")
print(df)

# Count the number of missing values in the 'sensor_reading' column
print("\nNumber of missing values:", df['sensor_reading'].isna().sum())

# Create a new column 'ffill_reading' by filling missing values using forward fill
df['ffill_reading'] = df['sensor_reading'].ffill()

# Create a new column 'interp_reading' by filling missing values using linear interpolation
df['interp_reading'] = df['sensor_reading'].interpolate(method='linear')

print("\nData after filling missing values:")
print(df)

# Convert the DatetimeIndex from 'America/New_York' to 'Europe/London'
df_london = df.tz_convert('Europe/London')
print("\nSensor data in London time:")
print(df_london)

# Print the first timestamp in both New York and London time
print("\nFirst timestamp - New York:", df.index[0])
print("First timestamp - London: ", df_london.index[0])


Original sensor data with missing values:
                           sensor_reading
2024-06-01 00:00:00-04:00            23.1
2024-06-01 01:00:00-04:00             NaN
2024-06-01 02:00:00-04:00            24.5
2024-06-01 03:00:00-04:00             NaN
2024-06-01 04:00:00-04:00             NaN
2024-06-01 05:00:00-04:00            26.0
2024-06-01 06:00:00-04:00            25.8
2024-06-01 07:00:00-04:00             NaN
2024-06-01 08:00:00-04:00            24.9
2024-06-01 09:00:00-04:00            23.7
2024-06-01 10:00:00-04:00             NaN
2024-06-01 11:00:00-04:00            22.5

Number of missing values: 5

Data after filling missing values:
                           sensor_reading  ffill_reading  interp_reading
2024-06-01 00:00:00-04:00            23.1           23.1           23.10
2024-06-01 01:00:00-04:00             NaN           23.1           23.80
2024-06-01 02:00:00-04:00            24.5           24.5           24.50
2024-06-01 03:00:00-04:00             NaN           24.